# 01. Supervised Baseline

This notebook trains a standard supervised benchmark. It treats business cards as `1` and consumer cards as `0` only for reference.

This is not the final modeling assumption, because consumer cards may contain hidden business users. The predicted probability is saved as `supervised_business_prob` and will be compared with PU-learning scores later.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from mdq.data import PROJECT_ROOT

pd.options.display.float_format = "{:,.4f}".format

RANDOM_STATE = 42
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [2]:
features = pd.read_parquet(PROCESSED_DIR / "card_features.parquet", engine="fastparquet")

features.shape, features["segment"].value_counts().to_dict()

((105000, 54), {'consumer': 80000, 'business': 25000})

## Feature Sets

We compare a behavior-only feature set with a full feature set that also includes detailed B2B MCC ratios. This helps check whether the model is learning broad behavior or mostly MCC-specific rules.

In [3]:
ID_COLS = ["card_number", "segment", "label"]

all_feature_cols = [c for c in features.columns if c not in ID_COLS]

mcc_group_cols = [
    c for c in all_feature_cols
    if c.startswith("mcc_") and c.endswith("_ratio")
]

behavior_feature_cols = [c for c in all_feature_cols if c not in mcc_group_cols]

feature_sets = {
    "behavior": behavior_feature_cols,
    "full": all_feature_cols,
}

{name: len(cols) for name, cols in feature_sets.items()}

{'behavior': 35, 'full': 51}

## Train / Validation / Test Split

The split is card-level because the feature table already has one row per card. Validation is used for model and threshold selection. Test is used once for the final baseline report.

In [4]:
train_idx, temp_idx = train_test_split(
    features.index,
    test_size=0.40,
    stratify=features["label"],
    random_state=RANDOM_STATE,
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=features.loc[temp_idx, "label"],
    random_state=RANDOM_STATE,
)

features = features.copy()
features["split"] = ""
features.loc[train_idx, "split"] = "train"
features.loc[val_idx, "split"] = "val"
features.loc[test_idx, "split"] = "test"

features.groupby(["split", "segment"]).size().unstack(fill_value=0)

segment,business,consumer
split,,
test,5000,16000
train,15000,48000
val,5000,16000


In [5]:
features.to_parquet(PROCESSED_DIR / "card_features_with_split.parquet", index=False, engine="fastparquet")

In [6]:
def evaluate_scores(y_true, score, threshold=0.5):
    pred = (score >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, score),
        "pr_auc": average_precision_score(y_true, score),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


def best_threshold_by_f1(y_true, score):
    thresholds = np.linspace(0.05, 0.95, 181)
    rows = []
    for threshold in thresholds:
        pred = (score >= threshold).astype(int)
        rows.append({
            "threshold": threshold,
            "f1": f1_score(y_true, pred, zero_division=0),
            "precision": precision_score(y_true, pred, zero_division=0),
            "recall": recall_score(y_true, pred, zero_division=0),
        })
    return pd.DataFrame(rows).sort_values("f1", ascending=False).iloc[0]


def make_logistic_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)),
    ])


def make_random_forest(max_depth=None, min_samples_leaf=5):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=250,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])

## Model Comparison On Validation

We compare two model families and two feature sets. Random Forest also gets a small parameter comparison through `max_depth` and `min_samples_leaf`.

In [7]:
train = features[features["split"] == "train"]
val = features[features["split"] == "val"]
test = features[features["split"] == "test"]

experiments = []

for feature_set_name, cols in feature_sets.items():
    experiments.append({
        "model_name": "logistic_regression",
        "feature_set": feature_set_name,
        "model": make_logistic_model(),
        "features": cols,
    })

    for max_depth, min_samples_leaf in [(8, 10), (12, 5), (None, 5)]:
        experiments.append({
            "model_name": f"random_forest_depth_{max_depth}_leaf_{min_samples_leaf}",
            "feature_set": feature_set_name,
            "model": make_random_forest(max_depth=max_depth, min_samples_leaf=min_samples_leaf),
            "features": cols,
        })

comparison_rows = []
fitted_models = {}

for exp in experiments:
    X_train = train[exp["features"]]
    y_train = train["label"]
    X_val = val[exp["features"]]
    y_val = val["label"]

    model = exp["model"]
    model.fit(X_train, y_train)
    val_score = model.predict_proba(X_val)[:, 1]
    threshold_row = best_threshold_by_f1(y_val.to_numpy(), val_score)
    metrics = evaluate_scores(y_val.to_numpy(), val_score, threshold=threshold_row["threshold"])

    key = (exp["model_name"], exp["feature_set"])
    fitted_models[key] = model

    comparison_rows.append({
        "model_name": exp["model_name"],
        "feature_set": exp["feature_set"],
        "n_features": len(exp["features"]),
        "best_threshold": threshold_row["threshold"],
        **{f"val_{k}": v for k, v in metrics.items()},
    })

model_comparison = pd.DataFrame(comparison_rows).sort_values(["val_f1", "val_pr_auc"], ascending=False)
model_comparison.to_csv(PROCESSED_DIR / "baseline_model_comparison.csv", index=False)
model_comparison

,model_name,feature_set,n_features,best_threshold,val_roc_auc,val_pr_auc,val_precision,val_recall,val_f1
4,logistic_regression,full,51,0.7800,1.0000,1.0000,0.9998,1.0000,0.9999
0,logistic_regression,behavior,35,0.8150,1.0000,1.0000,0.9998,0.9998,0.9998
3,random_forest_depth_None_leaf_5,behavior,35,0.4900,1.0000,1.0000,0.9996,0.9998,0.9997
2,random_forest_depth_12_leaf_5,behavior,35,0.5000,1.0000,1.0000,0.9996,0.9998,0.9997
7,random_forest_depth_None_leaf_5,full,51,0.5500,1.0000,1.0000,0.9996,0.9996,0.9996
6,random_forest_depth_12_leaf_5,full,51,0.4150,1.0000,1.0000,0.9990,1.0000,0.9995
5,random_forest_depth_8_leaf_10,full,51,0.5600,1.0000,1.0000,0.9992,0.9996,0.9994
1,random_forest_depth_8_leaf_10,behavior,35,0.6400,1.0000,1.0000,0.9994,0.9988,0.9991


## Final Baseline Test Metrics

In [8]:
best = model_comparison.iloc[0]
best_key = (best["model_name"], best["feature_set"])
best_features = feature_sets[best["feature_set"]]
best_threshold = best["best_threshold"]
best_model = fitted_models[best_key]

test_score = best_model.predict_proba(test[best_features])[:, 1]
test_metrics = evaluate_scores(test["label"].to_numpy(), test_score, threshold=best_threshold)
test_pred = (test_score >= best_threshold).astype(int)
test_cm = confusion_matrix(test["label"], test_pred)

baseline_test_metrics = pd.DataFrame([{
    "model_name": best["model_name"],
    "feature_set": best["feature_set"],
    "threshold": best_threshold,
    **test_metrics,
}])

baseline_test_metrics.to_csv(PROCESSED_DIR / "baseline_test_metrics.csv", index=False)

baseline_test_metrics, test_cm

(            model_name feature_set  threshold  roc_auc  pr_auc  precision  \
 0  logistic_regression        full     0.7800   1.0000  1.0000     0.9998   
 
    recall     f1  
 0  0.9998 0.9998  ,
 array([[15999,     1],
        [    1,  4999]]))

## Metric Meaning

- `ROC-AUC`: how well the model ranks business cards above consumer cards across all thresholds.
- `PR-AUC`: precision-recall area, more useful when the positive class is smaller.
- `precision`: among cards predicted as business, how many are actually labeled business in this baseline setup.
- `recall`: among labeled business cards, how many the model found.
- `F1`: balance between precision and recall.
- `confusion matrix`: exact counts of true/false predictions at the selected threshold.

Important: these metrics are for the supervised benchmark only. Consumer cards are treated as negatives here for comparison, but in the real task they are unlabeled.


## Confusion Matrix


In [9]:
confusion_matrix_df = pd.DataFrame(
    test_cm,
    index=["actual_consumer", "actual_business"],
    columns=["pred_consumer", "pred_business"],
)

confusion_matrix_df.to_csv(PROCESSED_DIR / "baseline_confusion_matrix.csv")
confusion_matrix_df


,pred_consumer,pred_business
actual_consumer,15999,1
actual_business,1,4999


## Feature Importance

For Random Forest we use built-in feature importance. For Logistic Regression we use absolute standardized coefficients.

In [10]:
if "random_forest" in best["model_name"]:
    importance = best_model.named_steps["model"].feature_importances_
else:
    importance = np.abs(best_model.named_steps["model"].coef_[0])

feature_importance = (
    pd.DataFrame({"feature": best_features, "importance": importance})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance.head(40)

,feature,importance
0,tokenized_ratio,4.3254
1,online_ratio,2.0164
2,recurring_ratio,1.5459
3,mcc_hhi,1.2631
4,merchant_hhi,1.2413
5,evening_ratio,1.2344
6,weekday_business_hours_ratio,1.1665
7,weekend_ratio,1.0305
8,mcc_advertising_ratio,0.9688
9,unique_merchant_countries,0.9563


## Save Baseline Probabilities

The best validation model is used to score every card. These probabilities are a supervised reference score, not the final PU score.

In [11]:
all_score = best_model.predict_proba(features[best_features])[:, 1]

baseline_scores = features[["card_number", "segment", "label", "split"]].copy()
baseline_scores["supervised_business_prob"] = all_score

baseline_scores.to_parquet(PROCESSED_DIR / "baseline_scores.parquet", index=False, engine="fastparquet")

baseline_consumer_scores = baseline_scores[baseline_scores["segment"] == "consumer"].copy()
baseline_consumer_scores.to_parquet(PROCESSED_DIR / "baseline_consumer_scores.parquet", index=False, engine="fastparquet")

baseline_scores.groupby("segment")["supervised_business_prob"].describe()

,count,mean,std,min,25%,50%,75%,max
segment,,,,,,,,
business,"25,000.0000",0.9996,0.0091,0.1129,1.0000,1.0000,1.0000,1.0000
consumer,"80,000.0000",0.0006,0.0137,0.0000,0.0000,0.0000,0.0000,0.9975


## Baseline Multi-Method Validation

This is the same idea as in the reference notebook, but kept as a baseline check. Supervised probability is compared with two unsupervised signals: consumer anomaly score and distance to the business centroid.

The purpose is not to replace PU learning. It is to show whether the supervised baseline finds candidates that also look unusual and business-like by independent checks.

In [12]:
def add_baseline_multi_method_scores(feature_table, feature_cols):
    consumer_part = feature_table[feature_table["segment"] == "consumer"].copy()
    business_part = feature_table[feature_table["segment"] == "business"].copy()

    consumer_preprocess = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    consumer_scaled = consumer_preprocess.fit_transform(consumer_part[feature_cols])

    iso = IsolationForest(
        n_estimators=250,
        contamination=0.01,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    iso.fit(consumer_scaled)
    consumer_part["baseline_anomaly_score"] = -iso.score_samples(consumer_scaled)

    business_preprocess = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    business_scaled = business_preprocess.fit_transform(business_part[feature_cols])
    consumer_scaled_to_business = business_preprocess.transform(consumer_part[feature_cols])
    business_centroid = business_scaled.mean(axis=0)

    consumer_part["baseline_distance_to_business"] = np.linalg.norm(
        consumer_scaled_to_business - business_centroid,
        axis=1,
    )

    consumer_part["baseline_supervised_rank"] = consumer_part["supervised_business_prob"].rank(pct=True)
    consumer_part["baseline_anomaly_rank"] = consumer_part["baseline_anomaly_score"].rank(pct=True)
    consumer_part["baseline_distance_rank"] = (-consumer_part["baseline_distance_to_business"]).rank(pct=True)
    consumer_part["baseline_ensemble_score"] = consumer_part[[
        "baseline_supervised_rank",
        "baseline_anomaly_rank",
        "baseline_distance_rank",
    ]].mean(axis=1)

    return consumer_part[[
        "card_number",
        "supervised_business_prob",
        "baseline_anomaly_score",
        "baseline_distance_to_business",
        "baseline_supervised_rank",
        "baseline_anomaly_rank",
        "baseline_distance_rank",
        "baseline_ensemble_score",
    ]]


baseline_multi_method_scores = add_baseline_multi_method_scores(
    features.merge(baseline_scores[["card_number", "supervised_business_prob"]], on="card_number", how="left"),
    best_features,
)
baseline_multi_method_scores.to_csv(PROCESSED_DIR / "baseline_multi_method_scores.csv", index=False)
baseline_multi_method_scores.sort_values("baseline_ensemble_score", ascending=False).head(20)

,card_number,supervised_business_prob,baseline_anomaly_score,baseline_distance_to_business,baseline_supervised_rank,baseline_anomaly_rank,baseline_distance_rank,baseline_ensemble_score
71890,5368296289935651,0.2045,0.6744,11.4331,0.9995,0.9934,0.9985,0.9971
70258,5368291791007503,0.3697,0.6707,11.1358,0.9997,0.9924,0.9991,0.9971
68766,5338479914183213,0.4325,0.6943,12.6120,0.9997,0.9982,0.9918,0.9966
67328,5338474007563215,0.9026,0.6600,6.9246,0.9999,0.9893,1.0000,0.9964
77969,5438812366882479,0.7737,0.6600,11.1529,0.9999,0.9893,0.9991,0.9961
37645,5176476550967656,0.1586,0.6625,11.6426,0.9993,0.9899,0.9977,0.9957
83076,5441020944586282,0.6716,0.6674,12.1760,0.9999,0.9912,0.9952,0.9954
80961,5438816990479651,0.7615,0.6477,11.0932,0.9999,0.9868,0.9992,0.9953
37681,5176476691114937,0.9257,0.6406,7.4361,1.0000,0.9858,1.0000,0.9952
27069,5100613360875775,0.0316,0.6536,11.0747,0.9979,0.9880,0.9993,0.9951


## Feature Diagnostics

This block checks whether features are actually useful or just duplicated/noisy. We do not rely on one model only. The table combines distribution checks, single-feature signal, correlation, Logistic Regression coefficients, and Random Forest importance.


In [13]:
diagnostic_features = all_feature_cols

business_part = features[features["segment"] == "business"]
consumer_part = features[features["segment"] == "consumer"]

diagnostic_rows = []
for col in diagnostic_features:
    values = features[col]
    business_values = business_part[col]
    consumer_values = consumer_part[col]

    auc = roc_auc_score(features["label"], values)
    single_feature_auc = max(auc, 1 - auc)
    business_median = business_values.median()
    consumer_median = consumer_values.median()
    pooled_std = values.std()
    median_gap_std = 0 if pooled_std == 0 else abs(business_median - consumer_median) / pooled_std

    diagnostic_rows.append({
        "feature": col,
        "nunique": values.nunique(),
        "std": pooled_std,
        "business_median": business_median,
        "consumer_median": consumer_median,
        "median_gap_std": median_gap_std,
        "single_feature_auc": single_feature_auc,
    })

feature_diagnostics = pd.DataFrame(diagnostic_rows)
feature_diagnostics.sort_values("single_feature_auc", ascending=False).head(20)


,feature,nunique,std,business_median,consumer_median,median_gap_std,single_feature_auc
13,online_ratio,8743,0.1861,0.8525,0.4472,2.1784,0.9877
19,evening_ratio,5913,0.1024,0.1071,0.3167,2.0458,0.9843
15,tokenized_ratio,6592,0.1015,0.5949,0.3855,2.0620,0.9819
18,weekend_ratio,6118,0.1087,0.1241,0.3502,2.0800,0.9802
17,weekday_business_hours_ratio,7120,0.1216,0.6429,0.3947,2.0402,0.9705
49,mcc_b2b_total_ratio,10706,0.2250,0.4604,0.0612,1.7738,0.9559
14,recurring_ratio,575,0.0745,0.1343,0.0000,1.8032,0.9495
50,mcc_b2b_total_amount_ratio,96510,0.2875,0.5862,0.0132,1.9929,0.9486
11,large_txn_100000_ratio,9783,0.2035,0.4557,0.0549,1.9693,0.9481
22,foreign_merchant_ratio,9727,0.2013,0.3231,0.0242,1.4842,0.9425


In [14]:
corr = features[diagnostic_features].corr().abs()
np.fill_diagonal(corr.values, 0)

max_corr = corr.max().rename("max_abs_corr")
corr_with = corr.idxmax().rename("most_correlated_with")

feature_diagnostics = (
    feature_diagnostics
    .merge(max_corr, left_on="feature", right_index=True)
    .merge(corr_with, left_on="feature", right_index=True)
)

feature_diagnostics.sort_values("max_abs_corr", ascending=False).head(20)


,feature,nunique,std,business_median,consumer_median,median_gap_std,single_feature_auc,max_abs_corr,most_correlated_with
32,mcc_hhi,70352,0.1017,0.1930,0.0519,1.3873,0.8978,0.9990,merchant_hhi
27,merchant_hhi,68902,0.1038,0.1921,0.0455,1.4121,0.8972,0.9990,mcc_hhi
33,top_mcc_share,7528,0.1504,0.3712,0.1188,1.6780,0.8980,0.9965,top_merchant_share
28,top_merchant_share,7402,0.1533,0.3707,0.1111,1.6932,0.8982,0.9965,top_mcc_share
26,merchant_entropy,103223,0.7305,2.0962,3.3461,1.7111,0.9014,0.9926,mcc_entropy
31,mcc_entropy,103753,0.6708,2.0731,3.1998,1.6798,0.9049,0.9926,merchant_entropy
3,log_avg_amount,104976,1.0347,11.9551,10.1953,1.7009,0.9164,0.9925,log_amount_per_active_day
9,log_amount_per_active_day,104985,1.0518,12.3443,10.5107,1.7432,0.9144,0.9925,log_avg_amount
6,log_p90_amount,102306,1.0843,12.8212,11.0162,1.6646,0.9210,0.9834,log_avg_amount
34,top_mcc_amount_share,105000,0.1770,0.3691,0.2446,0.7031,0.7213,0.9800,top_merchant_amount_share


In [15]:
full_train_cols = feature_sets["full"]

diagnostic_logreg = make_logistic_model()
diagnostic_logreg.fit(train[full_train_cols], train["label"])
logreg_importance = pd.Series(
    np.abs(diagnostic_logreg.named_steps["model"].coef_[0]),
    index=full_train_cols,
    name="logreg_abs_coef",
)

diagnostic_rf = make_random_forest(max_depth=12, min_samples_leaf=5)
diagnostic_rf.fit(train[full_train_cols], train["label"])
rf_importance = pd.Series(
    diagnostic_rf.named_steps["model"].feature_importances_,
    index=full_train_cols,
    name="rf_importance",
)

feature_diagnostics = (
    feature_diagnostics
    .merge(logreg_importance, left_on="feature", right_index=True, how="left")
    .merge(rf_importance, left_on="feature", right_index=True, how="left")
)

feature_diagnostics[["logreg_abs_coef", "rf_importance"]] = feature_diagnostics[["logreg_abs_coef", "rf_importance"]].fillna(0)
feature_diagnostics.sort_values("rf_importance", ascending=False).head(20)


,feature,nunique,std,business_median,consumer_median,median_gap_std,single_feature_auc,max_abs_corr,most_correlated_with,logreg_abs_coef,rf_importance
19,evening_ratio,5913,0.1024,0.1071,0.3167,2.0458,0.9843,0.8478,online_ratio,1.2344,0.1495
18,weekend_ratio,6118,0.1087,0.1241,0.3502,2.0800,0.9802,0.9121,weekday_business_hours_ratio,1.0305,0.1300
13,online_ratio,8743,0.1861,0.8525,0.4472,2.1784,0.9877,0.8478,evening_ratio,2.0164,0.1119
50,mcc_b2b_total_amount_ratio,96510,0.2875,0.5862,0.0132,1.9929,0.9486,0.9435,mcc_b2b_total_ratio,0.4851,0.1102
17,weekday_business_hours_ratio,7120,0.1216,0.6429,0.3947,2.0402,0.9705,0.9121,weekend_ratio,1.1665,0.0769
49,mcc_b2b_total_ratio,10706,0.2250,0.4604,0.0612,1.7738,0.9559,0.9435,mcc_b2b_total_amount_ratio,0.1009,0.0683
15,tokenized_ratio,6592,0.1015,0.5949,0.3855,2.0620,0.9819,0.7450,weekday_business_hours_ratio,4.3254,0.0549
22,foreign_merchant_ratio,9727,0.2013,0.3231,0.0242,1.4842,0.9425,0.9248,recurring_capable_ratio,0.0094,0.0439
31,mcc_entropy,103753,0.6708,2.0731,3.1998,1.6798,0.9049,0.9926,merchant_entropy,0.0092,0.0407
27,merchant_hhi,68902,0.1038,0.1921,0.0455,1.4121,0.8972,0.9990,mcc_hhi,1.2413,0.0352


In [16]:
low_auc = feature_diagnostics["single_feature_auc"] < 0.55
low_gap = feature_diagnostics["median_gap_std"] < 0.10
low_logreg = feature_diagnostics["logreg_abs_coef"] < feature_diagnostics["logreg_abs_coef"].quantile(0.25)
low_rf = feature_diagnostics["rf_importance"] < feature_diagnostics["rf_importance"].quantile(0.25)
near_duplicate = feature_diagnostics["max_abs_corr"] > 0.97
almost_constant = (feature_diagnostics["nunique"] <= 2) | (feature_diagnostics["std"] < 1e-9)

feature_diagnostics["diagnostic_flag"] = "keep"
feature_diagnostics.loc[near_duplicate, "diagnostic_flag"] = "review_duplicate"
feature_diagnostics.loc[low_auc & low_gap & low_logreg & low_rf, "diagnostic_flag"] = "drop_candidate"
feature_diagnostics.loc[almost_constant, "diagnostic_flag"] = "drop_candidate"

def diagnostic_reason(row):
    reasons = []
    if row["nunique"] <= 2 or row["std"] < 1e-9:
        reasons.append("almost constant")
    if row["single_feature_auc"] < 0.55:
        reasons.append("weak single-feature AUC")
    if row["median_gap_std"] < 0.10:
        reasons.append("small median gap")
    if row["max_abs_corr"] > 0.97:
        reasons.append(f"near-duplicate of {row['most_correlated_with']}")
    if row["logreg_abs_coef"] < feature_diagnostics["logreg_abs_coef"].quantile(0.25):
        reasons.append("low logistic weight")
    if row["rf_importance"] < feature_diagnostics["rf_importance"].quantile(0.25):
        reasons.append("low RF importance")
    return "; ".join(reasons)

feature_diagnostics["diagnostic_reason"] = feature_diagnostics.apply(diagnostic_reason, axis=1)

feature_diagnostics = feature_diagnostics.sort_values(
    ["diagnostic_flag", "single_feature_auc", "rf_importance"],
    ascending=[True, True, True],
)

feature_diagnostics.to_csv(PROCESSED_DIR / "baseline_feature_diagnostics.csv", index=False)

feature_diagnostics[feature_diagnostics["diagnostic_flag"] != "keep"]


,feature,nunique,std,business_median,consumer_median,median_gap_std,single_feature_auc,max_abs_corr,most_correlated_with,logreg_abs_coef,rf_importance,diagnostic_flag,diagnostic_reason
0,txn_count,317,46.8684,119.0000,120.0000,0.0213,0.5084,0.9697,active_days,0.0467,0.0002,drop_candidate,weak single-feature AUC; small median gap; low...
34,top_mcc_amount_share,105000,0.1770,0.3691,0.2446,0.7031,0.7213,0.9800,top_merchant_amount_share,0.3970,0.0024,review_duplicate,near-duplicate of top_merchant_amount_share
29,top_merchant_amount_share,105000,0.1796,0.3688,0.2266,0.7916,0.7481,0.9800,top_mcc_amount_share,0.3170,0.0022,review_duplicate,near-duplicate of top_mcc_amount_share
27,merchant_hhi,68902,0.1038,0.1921,0.0455,1.4121,0.8972,0.9990,mcc_hhi,1.2413,0.0352,review_duplicate,near-duplicate of mcc_hhi
32,mcc_hhi,70352,0.1017,0.1930,0.0519,1.3873,0.8978,0.9990,merchant_hhi,1.2631,0.0249,review_duplicate,near-duplicate of merchant_hhi
33,top_mcc_share,7528,0.1504,0.3712,0.1188,1.6780,0.8980,0.9965,top_merchant_share,0.1246,0.0113,review_duplicate,near-duplicate of top_merchant_share
28,top_merchant_share,7402,0.1533,0.3707,0.1111,1.6932,0.8982,0.9965,top_mcc_share,0.0644,0.0227,review_duplicate,near-duplicate of top_mcc_share; low logistic ...
26,merchant_entropy,103223,0.7305,2.0962,3.3461,1.7111,0.9014,0.9926,mcc_entropy,0.0021,0.0124,review_duplicate,near-duplicate of mcc_entropy; low logistic we...
31,mcc_entropy,103753,0.6708,2.0731,3.1998,1.6798,0.9049,0.9926,merchant_entropy,0.0092,0.0407,review_duplicate,near-duplicate of merchant_entropy; low logist...
9,log_amount_per_active_day,104985,1.0518,12.3443,10.5107,1.7432,0.9144,0.9925,log_avg_amount,0.0448,0.0154,review_duplicate,near-duplicate of log_avg_amount; low logistic...


The diagnostic flag is not an automatic deletion rule. A feature should be removed only if the statistical checks and business logic agree. Near-duplicates are reviewed manually, because one of the pair may be more interpretable or more stable.


## Top Consumer Candidates And Reasons

Now we inspect consumer cards with the highest supervised business probability. This is only a baseline view, but it helps check whether high-score cards have business-like behavior.


In [17]:
profile_cols = [
    "log_total_spend",
    "log_avg_amount",
    "log_median_amount",
    "log_p90_amount",
    "amount_cv",
    "online_ratio",
    "recurring_ratio",
    "recurring_capable_ratio",
    "weekday_business_hours_ratio",
    "weekend_ratio",
    "evening_ratio",
    "foreign_merchant_ratio",
    "unique_merchants",
    "unique_mcc",
    "top_merchant_share",
    "top_mcc_share",
    "mcc_b2b_total_ratio",
    "mcc_b2b_total_amount_ratio",
]

business_median = features[features["segment"] == "business"][profile_cols].median()
consumer_median = features[features["segment"] == "consumer"][profile_cols].median()

def candidate_reasons(row):
    reasons = []
    if row["online_ratio"] >= business_median["online_ratio"]:
        reasons.append("online ratio like business")
    if row["recurring_ratio"] >= business_median["recurring_ratio"]:
        reasons.append("high recurring payments")
    if row["foreign_merchant_ratio"] >= business_median["foreign_merchant_ratio"]:
        reasons.append("high foreign merchant exposure")
    if row["mcc_b2b_total_ratio"] >= business_median["mcc_b2b_total_ratio"]:
        reasons.append("high B2B MCC transaction share")
    if row["mcc_b2b_total_amount_ratio"] >= business_median["mcc_b2b_total_amount_ratio"]:
        reasons.append("high B2B MCC spend share")
    if row["weekday_business_hours_ratio"] >= business_median["weekday_business_hours_ratio"]:
        reasons.append("work-hours heavy")
    if row["weekend_ratio"] <= business_median["weekend_ratio"]:
        reasons.append("low weekend usage")
    if row["evening_ratio"] <= business_median["evening_ratio"]:
        reasons.append("low evening usage")
    if row["log_total_spend"] >= business_median["log_total_spend"]:
        reasons.append("business-level total spend")
    return "; ".join(reasons[:5])


In [18]:
top_consumer_candidates = (
    baseline_scores[baseline_scores["segment"] == "consumer"]
    .sort_values("supervised_business_prob", ascending=False)
    .head(50)
    .merge(features[["card_number"] + profile_cols], on="card_number", how="left")
)

top_consumer_candidates["baseline_reasons"] = top_consumer_candidates.apply(candidate_reasons, axis=1)

top_consumer_candidates.to_csv(PROCESSED_DIR / "baseline_top_consumer_candidates.csv", index=False)

top_consumer_candidates[[
    "card_number",
    "supervised_business_prob",
    "baseline_reasons",
    "online_ratio",
    "recurring_ratio",
    "foreign_merchant_ratio",
    "mcc_b2b_total_ratio",
    "mcc_b2b_total_amount_ratio",
    "weekday_business_hours_ratio",
    "weekend_ratio",
    "evening_ratio",
    "log_total_spend",
]].head(20)


,card_number,supervised_business_prob,baseline_reasons,online_ratio,recurring_ratio,foreign_merchant_ratio,mcc_b2b_total_ratio,mcc_b2b_total_amount_ratio,weekday_business_hours_ratio,weekend_ratio,evening_ratio,log_total_spend
0,5201491354169846,0.9975,online ratio like business; high recurring pay...,0.8846,0.2308,0.4423,0.2692,0.3806,0.6538,0.0769,0.0769,16.4282
1,5228597629027905,0.9729,online ratio like business; high recurring pay...,0.8537,0.4390,0.7561,0.6829,0.9071,0.3415,0.1951,0.0488,16.5927
2,5500442485584260,0.9728,online ratio like business; high recurring pay...,0.8654,0.3462,0.8077,0.7115,0.9811,0.5000,0.1731,0.0962,17.6052
3,5100612020402608,0.9699,high recurring payments; work-hours heavy; low...,0.8276,0.1552,0.3017,0.2241,0.5391,0.6810,0.1121,0.1121,16.3873
4,5176476691114937,0.9257,online ratio like business; high recurring pay...,0.8701,0.2338,0.3636,0.3506,0.6644,0.5714,0.0909,0.1169,16.0171
5,5338474007563215,0.9026,high recurring payments; high foreign merchant...,0.8495,0.1935,0.5591,0.4839,0.4629,0.6774,0.0968,0.0753,16.5119
6,5176513825363681,0.8583,online ratio like business; high recurring pay...,0.8554,0.2169,0.3373,0.2410,0.4171,0.6024,0.1928,0.0602,16.0598
7,5176516958585590,0.8503,online ratio like business; high recurring pay...,0.8986,0.2609,0.7536,0.7391,0.8678,0.5797,0.1159,0.0580,16.8381
8,5438812366882479,0.7737,high recurring payments; high foreign merchant...,0.7778,0.1481,0.7407,0.8148,0.8081,0.7284,0.0617,0.1111,16.3684
9,5438816990479651,0.7615,work-hours heavy; low weekend usage; low eveni...,0.7765,0.0000,0.0000,0.3294,0.3997,0.8588,0.0235,0.0588,17.2133


The `baseline_reasons` column is a simple human-readable rule layer. It does not replace SHAP or PU explainability, but it makes the supervised baseline easy to inspect by hand.


## Baseline Note

The baseline is useful as a benchmark, but high metrics do not fully solve the task. The consumer set may contain hidden business users, so the next step is PU learning and ranking.

In [19]:
features[features['card_number'] == '5201491354169846']

,card_number,segment,label,txn_count,amount_cv,log_total_spend,log_avg_amount,log_median_amount,log_std_amount,log_p90_amount,...,mcc_office_supplies_amount_ratio,mcc_telecom_ratio,mcc_telecom_amount_ratio,mcc_logistics_ratio,mcc_logistics_amount_ratio,mcc_professional_services_ratio,mcc_professional_services_amount_ratio,mcc_b2b_total_ratio,mcc_b2b_total_amount_ratio,split
50695,5201491354169846,consumer,0,52,1.5709,16.4282,12.4770,11.5373,12.9286,13.6688,...,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.2692,0.3806,train
